##### ARTI 560 - Computer Vision  

## Pose Estimation - Exercise  

### Objective  

In this exercise, you will explore **human pose estimation** by running two different state-of-the-art models: **YOLOv7 Pose** and **OpenPose**. You will apply both models to video data and compare the output.

The goal is to understand how different pose estimation approaches work in practice and how model design affects performance in real-world scenarios.

### YOLOv7 Pose Estimation

**YOLOv7 Pose** extends the YOLO (You Only Look Once) object detection framework to perform **real-time multi-person pose estimation**.  

Key characteristics:
- Uses a **single-stage architecture** (detects people and keypoints in one pass).
- Fast and suitable for **real-time applications**.
- Outputs bounding boxes along with **keypoint coordinates**.
- Efficient for videos with multiple people.

Unlike traditional pipelines that first detect people and then estimate poses separately, YOLOv7 integrates both steps into one unified model, improving speed and efficiency.


### Tasks  

You'll run both taks 1 and task 2 on a video from your choice from the media folder.

#### Task 1: Run YOLOv7 Pose Estimation  

1. Navigate to the yolov7 project directory.

2. Create a new Conda environment with Python 3.9:
   ```bash
   conda create -n yolov7_pose python=3.9
   conda activate yolov7_pose
   ```

3. Install the required dependencies:
   ```bash
   pip install -r requirements.txt
   ```

4. Run the YOLOv7 pose estimation script:
   ```bash
   python yolov7-pose.py
   ```
   *Note you must modify the script to update the video path accordingly.


#### Task 2: Run OpenPose  

1. Navigate to the OpenPose project directory.

2. Run the OpenPose video script:
   ```bash
   python OpenPoseVideo.py --video_file "path_to_video"
   ```
   *Note: Replace "path_to_video" with the correct path to a video from the media folder.

  

### Submission Requirements  
- Include **screenshots showing the successful execution of both scripts** (YOLOv7 Pose and OpenPose), clearly displaying the terminal.
- Output videos from both models (**YOLOv7 Pose** and **OpenPose**) must be **pushed to your GitHub repository**.  

## Solution

For this exercise, I use `walking-persons.mp4` from the `media` folder because it contains multiple people walking, which is a useful case for comparing multi-person pose estimation.

The solution includes:

- a fixed YOLOv7 Pose script with command-line arguments and a correct output writer,
- a fixed OpenPose script with safer model/video path handling,
- commands to run both methods on the same video,
- output paths for the generated videos,
- a short comparison between the two approaches.


### 1. Define paths


In [1]:
from pathlib import Path
import subprocess
import sys

lab_dir = Path.cwd()
if not (lab_dir / "media").exists() and (lab_dir / "lab08-pose-estimation").exists():
    lab_dir = lab_dir / "lab08-pose-estimation"
media_dir = lab_dir / "media"
yolov7_dir = lab_dir / "yolov7"
openpose_dir = lab_dir / "pose"
output_dir = lab_dir / "outputs"

selected_video = media_dir / "walking-persons.mp4"
yolov7_weights = yolov7_dir / "yolov7-w6-pose.pt"
openpose_mpi_weights = openpose_dir / "mpi" / "pose_iter_160000.caffemodel"
openpose_coco_weights = openpose_dir / "coco" / "pose_iter_440000.caffemodel"

output_dir.mkdir(exist_ok=True)

print("Selected video:", selected_video)
print("Output folder:", output_dir)


Selected video: a:\lab08-pose-estimation\media\walking-persons.mp4
Output folder: a:\lab08-pose-estimation\outputs


### 2. Check required files

The video is already included in the repository. The model weight files are large, so they may need to be downloaded separately before running the scripts.


In [2]:
required_files = {
    "Input video": selected_video,
    "YOLOv7 pose weights": yolov7_weights,
    "OpenPose MPI weights": openpose_mpi_weights,
}

for label, path in required_files.items():
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{label}: {status} -> {path}")

if not yolov7_weights.exists():
    print("\nDownload YOLOv7 weights from:")
    print("https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-w6-pose.pt")
    print("Place the file inside the yolov7 folder.")

if not openpose_mpi_weights.exists():
    print("\nDownload OpenPose MPI weights from:")
    print("https://www.dropbox.com/s/drumc6dzllfed16/pose_iter_160000.caffemodel")
    print("Place the file inside pose/mpi.")


Input video: FOUND -> a:\lab08-pose-estimation\media\walking-persons.mp4
YOLOv7 pose weights: FOUND -> a:\lab08-pose-estimation\yolov7\yolov7-w6-pose.pt
OpenPose MPI weights: FOUND -> a:\lab08-pose-estimation\pose\mpi\pose_iter_160000.caffemodel


### 3. Task 1: Run YOLOv7 Pose

The YOLOv7 script was updated so the input video, weights, device, and output folder can be passed from the command line. The command below runs on CPU for compatibility.


In [3]:
yolov7_command = [
    sys.executable,
    str(yolov7_dir / "yolov7-pose.py"),
    "--video-file", str(selected_video),
    "--weights", str(yolov7_weights),
    "--output-dir", str(output_dir),
    "--device", "cpu",
    "--img-size", "256",
]

print("Run this command from the lab folder:")
print(" ".join(str(part) for part in yolov7_command))

if yolov7_weights.exists():
    result = subprocess.run(yolov7_command, cwd=lab_dir, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
else:
    print("Skipped YOLOv7 run because yolov7-w6-pose.pt is missing.")


Run this command from the lab folder:
a:\envs\tf210\python.exe a:\lab08-pose-estimation\yolov7\yolov7-pose.py --video-file a:\lab08-pose-estimation\media\walking-persons.mp4 --weights a:\lab08-pose-estimation\yolov7\yolov7-w6-pose.pt --output-dir a:\lab08-pose-estimation\outputs --device cpu --img-size 256
Selected device: cpu
Processed frames: 341
YOLOv7 Pose output saved to: a:\lab08-pose-estimation\outputs\walking-persons_yolov7_pose.avi

a:\envs\tf210\lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]



### 4. Task 2: Run OpenPose

The OpenPose script was updated to accept the video path, output folder, model type, and device from the command line. This run uses the MPI model because it is smaller and usually faster.


In [4]:
openpose_command = [
    sys.executable,
    "OpenPoseVideo.py",
    "--video_file", str(selected_video),
    "--output-dir", str(output_dir),
    "--model", "MPI",
    "--device", "cpu",
]

print("Run this command from the pose folder:")
print(" ".join(str(part) for part in openpose_command))

if openpose_mpi_weights.exists():
    result = subprocess.run(openpose_command, cwd=openpose_dir, text=True, capture_output=True)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
else:
    print("Skipped OpenPose run because pose_iter_160000.caffemodel is missing.")


Run this command from the pose folder:
a:\envs\tf210\python.exe OpenPoseVideo.py --video_file a:\lab08-pose-estimation\media\walking-persons.mp4 --output-dir a:\lab08-pose-estimation\outputs --model MPI --device cpu
Using CPU device
Processed frames: 341
OpenPose output saved to: a:\lab08-pose-estimation\outputs\walking-persons_openpose_mpi.avi



### 5. Expected output files


In [5]:
yolov7_output = output_dir / f"{selected_video.stem}_yolov7_pose.avi"
openpose_output = output_dir / f"{selected_video.stem}_openpose_mpi.avi"

for output_path in [yolov7_output, openpose_output]:
    status = "CREATED" if output_path.exists() else "NOT CREATED YET"
    print(f"{status}: {output_path}")


CREATED: a:\lab08-pose-estimation\outputs\walking-persons_yolov7_pose.avi
CREATED: a:\lab08-pose-estimation\outputs\walking-persons_openpose_mpi.avi


### 6. Comparison

| Model | Pipeline type | Strengths | Limitations |
|---|---|---|---|
| YOLOv7 Pose | Single-stage detection and keypoint estimation | Faster and better suited for real-time video; detects people and keypoints together | Requires the YOLOv7 pose weight file and PyTorch setup |
| OpenPose | Bottom-up keypoint detection and grouping | Strong multi-person pose-estimation concept; useful for understanding part affinity fields | Slower, needs Caffe model weights, and can be more sensitive to occlusion |

For `walking-persons.mp4`, YOLOv7 Pose is expected to run faster because it uses one unified network. OpenPose is useful for comparison because it follows a different design: it detects body parts and then connects them into people.


### Submission notes

After both scripts run successfully, include terminal screenshots showing:

- the YOLOv7 command and its successful output path,
- the OpenPose command and its successful output path.

Push these output videos to GitHub:

- `outputs/walking-persons_yolov7_pose.avi`
- `outputs/walking-persons_openpose_mpi.avi`
